# 네이버 리뷰 크롤링 단계별 테스트
각 셀을 순서대로 실행하세요. 드라이버는 열린 상태로 유지됩니다.

In [1]:
import sys, time, pickle, re
from pathlib import Path
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

ROOT         = Path("../")
CRAWLING_DIR = ROOT / "crawling"
COOKIES_DIR  = Path("./cookies")

sys.path.insert(0, str(CRAWLING_DIR))
from core.browser import create_driver, dismiss_popups, safe_click
from review_config import REVIEW_SELECTORS

sel = REVIEW_SELECTORS["naver"]
print("셀렉터 로드 완료")
print("review_btn  :", sel.get("review_btn"))
print("wait_selector:", sel.get("wait_selector"))
print("container   :", sel.get("container"))

셀렉터 로드 완료
review_btn  : #content > div > div.fUgLLODhD8 > div:nth-child(1) > div > div > div.PSOcMLEJuY > button
wait_selector: #MODAL_ROOT_ID > div > div.qc8qCgj4u2.b3VJJSdlmJ > div > div > div.ckqgS03UN6 > div > div > div:nth-child(1) > div > div.ZDSfEuYSzZ > strong
container   : .PYRRKjHPB6


In [2]:
# ── 드라이버 생성 + 쿠키 로드
driver = create_driver()

cookie_path = COOKIES_DIR / "naver.pkl"
if cookie_path.exists():
    driver.get("https://www.naver.com")
    for cookie in pickle.load(open(cookie_path, "rb")):
        try:
            driver.add_cookie(cookie)
        except Exception:
            pass
    print("쿠키 로드 완료")
else:
    print("쿠키 파일 없음 — 비로그인 상태로 진행")

쿠키 로드 완료


In [3]:
# ── 상품 페이지 이동 (URL 직접 입력)
URL = "https://smartstore.naver.com/malldaewoong/products/12727841644"  # 여기에 URL 입력

driver.get(URL)
time.sleep(3)
dismiss_popups(driver)
print("페이지 로드 완료:", driver.title)

페이지 로드 완료: 에너씨슬 집중샷 오렌지1박스+샤인머스캣 2박스 식물성카페인 아르기닌 스터디젤리 : 대웅제약 공식 브랜드스토어


In [4]:
# ── 배율 50% 축소
driver.execute_script("document.body.style.zoom='50%'")
time.sleep(0.5)
print("zoom 50% 적용")

zoom 50% 적용


In [5]:
# ── review_btn 요소 확인 및 scrollIntoView
try:
    btn = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, sel["review_btn"]))
    )
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
    print("버튼 발견:", btn.text)
except Exception as e:
    print("버튼 없음:", e)

버튼 발견: 리뷰 전체보기


In [6]:
# ── 리뷰전체보기 JS 클릭
try:
    btn = driver.find_element(By.CSS_SELECTOR, sel["review_btn"])
    driver.execute_script("arguments[0].click();", btn)
    print("리뷰전체보기 클릭 완료")
except Exception as e:
    print("클릭 실패:", e)

리뷰전체보기 클릭 완료


In [7]:
# ── 모달 로드 대기 (wait_selector)
try:
    WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, sel["wait_selector"]))
    )
    print("모달 로드 완료")
except Exception as e:
    print("모달 대기 실패:", e)

모달 로드 완료


In [8]:
# ── 무한스크롤 1회 테스트 (모달 컨테이너 기준)
SCROLL_SEL = sel.get("scroll_container", "")
print("scroll_container:", SCROLL_SEL)

before = driver.execute_script(f"var el=document.querySelector('{SCROLL_SEL}'); return el ? el.scrollHeight : 0;")
driver.execute_script(f"var el=document.querySelector('{SCROLL_SEL}'); if(el) el.scrollTop=el.scrollHeight;")
time.sleep(2)
after = driver.execute_script(f"var el=document.querySelector('{SCROLL_SEL}'); return el ? el.scrollHeight : 0;")
print(f"scrollHeight: {before} → {after} ({'증가' if after > before else '변화없음'})")

scroll_container: div.ckqgS03UN6
scrollHeight: 10570 → 20090 (증가)


In [9]:
# ── 무한스크롤 끝까지 (모달 컨테이너 기준)
SCROLL_SEL = sel.get("scroll_container", "")
last_height = 0
page = 0
while True:
    driver.execute_script(f"var el=document.querySelector('{SCROLL_SEL}'); if(el) el.scrollTop=el.scrollHeight;")
    time.sleep(2)
    new_height = driver.execute_script(f"var el=document.querySelector('{SCROLL_SEL}'); return el ? el.scrollHeight : 0;")
    page += 1
    print(f"  scroll {page}: height={new_height}")
    if new_height == last_height:
        print("스크롤 끝")
        break
    last_height = new_height

  scroll 1: height=29518
  scroll 2: height=39255
  scroll 3: height=49204
  scroll 4: height=58606
  scroll 5: height=68019
  scroll 6: height=77664
  scroll 7: height=86825
  scroll 8: height=96013
  scroll 9: height=105711
  scroll 10: height=115409
  scroll 11: height=124617
  scroll 12: height=134679
  scroll 13: height=144826
  scroll 14: height=154508
  scroll 15: height=163539
  scroll 16: height=173460
  scroll 17: height=183427
  scroll 18: height=193067
  scroll 19: height=203329
  scroll 20: height=214041
  scroll 21: height=223378
  scroll 22: height=232560
  scroll 23: height=242696
  scroll 24: height=252181
  scroll 25: height=262242
  scroll 26: height=272112
  scroll 27: height=281569
  scroll 28: height=290483
  scroll 29: height=299630
  scroll 30: height=308815
  scroll 31: height=317374
  scroll 32: height=326565
  scroll 33: height=336190
  scroll 34: height=345580
  scroll 35: height=355525
  scroll 36: height=365100
  scroll 37: height=374728
  scroll 38: heigh

In [10]:
# ── 리뷰 파싱 테스트
soup = BeautifulSoup(driver.page_source, "html.parser")
blocks = soup.select(sel["container"])
print(f"리뷰 블록 수: {len(blocks)}")

def extract(el, s):
    if not s: return ""
    found = el.select_one(s)
    return found.get_text(strip=True) if found else ""

reviews = []
for block in blocks:
    reviews.append({
        "review_date": extract(block, sel["date"]),
        "rating":      extract(block, sel["rating"]),
        "reviewer":    extract(block, sel["reviewer"]),
        "content":     extract(block, sel["content"]),
    })

print(f"\n수집된 리뷰 {len(reviews)}건 (앞 3건 미리보기):")
for r in reviews[:3]:
    print(f"  [{r['reviewer']}] {r['rating']}점 | {r['review_date']}")
    print(f"  {r['content'][:80]}")
    print()

리뷰 블록 수: 1133

수집된 리뷰 1133건 (앞 3건 미리보기):
  [pear****] 5점 | 26.04.10.
  유효기간 좋고 실온보관이라 좋네요. 에너씨슬 집중샷은 약 먹기 싫어하는 아이에게 간식같은 느낌으롳먹기 편한 젤리스틱이라 구입해 봅니다. 샤인머스

  [rhyc******] 5점 | 26.04.20.
  중3 딸래미 시험시간이라 필요할 때 먹으라고 구매했어요. 
올 해는 전교회장이라는 타이틀도 생겨
누구보다 더 잘해야겠다라는 부담과 강박을 
느끼

  [gmlw*****] 5점 | 26.04.14.
  졸음 참느라 커피만 들이키는 고등학생아이가 안쓰러워서 사봤습니다. 애들은 늘 피곤하니깐 드라마틱한 도움보단 좀 낫다고 하구요. 저는 장거리 운전



In [11]:
driver.quit()
print("드라이버 종료")

드라이버 종료
